# CytoMind Compensation Tutorial

This notebook demonstrates the **spillover compensation workflow** in CytoMind, including:
- Loading a prepared project repository with registered samples and loaded FCS data
- Applying compensation matrices to raw flow cytometry data
- Reviewing QC metrics for compensation quality
- Interactively revising compensation coefficients and visualizing results

## Prerequisites

Before running this notebook, you must have:
1. **Created a project repository** using `AddSamplesStep` to register FCS files and extract panel/compensation metadata
2. **Loaded FCS data** using `LoadFCS` step to convert FCS files to AnnData HDF5 format
3. **Prepared sample FCS files** containing compensation matrices in their metadata

See the **Setup Instructions** section below for how to prepare your data.

## Setup Instructions: Preparing Your Data

This notebook can work in **two modes**:

### Mode 1: Auto-Initialize Project (Recommended)

Provide FCS file paths in the **Configuration** section, and the notebook will automatically initialize the project:

```python
from pathlib import Path

# In Configuration section (cell 8), set:
PROJECT_REPO_PATH = Path("my_cytomind_project")
SAMPLE_IDS = {
    "sample_1": "path/to/sample_1.fcs",
    "sample_2": "path/to/sample_2.fcs",
}
```

The notebook will then automatically:
1. Create the project repository
2. Register samples and extract panel/compensation metadata
3. Load FCS data into AnnData format
4. Proceed with compensation workflow

### Mode 2: Use Existing Project Repository

If you've already initialized a project, simply provide the repository path:

```python
PROJECT_REPO_PATH = Path("my_cytomind_project")
SAMPLE_IDS = {}  # Empty dict - use samples already in project
```

### Prerequisites for FCS Files

Your FCS files should:
- Be in **Flow Cytometry Standard (FCS) format** (3.0 or later)
- Contain **spillover matrix metadata** (typically embedded by flow cytometer).
  If they do not, then compensation matrices must be provided by the user.
- Have **consistent channel panels** across samples (or be in separate batches)
  or a dictionary that specifies channel renamings.

### Manual Initialization (Alternative)

If you prefer to initialize the project manually before running this notebook:

```python
from pathlib import Path
from cytomind import InteractivePipeline

# Create a new project repository
repo_path = Path("my_cytomind_project")
pipeline = InteractivePipeline(repo_path)

# Add samples (sample_id -> FCS file path mapping)
pipeline.add_samples({
    "sample_1": "path/to/sample_1.fcs",
    "sample_2": "path/to/sample_2.fcs",
})

# Load FCS data into AnnData format
pipeline.load_fcs()
```

Then in this notebook, set `SAMPLE_IDS = {}` to use the pre-loaded project.

---

# Section 1: Setup Notebook Environment

In [1]:
# Enable autoreload to pick up local code changes
%load_ext autoreload
%autoreload 2

import cytomind
print(f"CytoMind version: {cytomind.__version__ if hasattr(cytomind, '__version__') else 'unknown'}")

CytoMind version: 0.1.0


## Section 2: Import Dependencies

In [2]:
from pathlib import Path
from shutil import rmtree
import numpy as np

import pandas as pd

import plotly
import plotly.graph_objects as go

# Configure plotly renderer for notebooks
plotly.io.renderers.default = "notebook_connected"

print("✓ All dependencies imported successfully")

✓ All dependencies imported successfully


## Section 3: Configuration

**EDIT THIS SECTION** with your project paths and compensation settings.

In [ ]:
# ============================================================================
# USER CONFIGURATION: Update these paths for your data
# ============================================================================

# Path to the CytoMind project repository
PROJECT_REPO_PATH = Path("./results/compensation_tutorial")

# Sample mapping: {'sample_id': 'path/to/fcs'}
# If empty dict {}, the notebook will use samples already in the project
# If dict is provided and project is not initialized, it will run add_samples + load_fcs
# Example: SAMPLE_IDS = {"sample_1": "data/sample_1.fcs", "sample_2": "data/sample_2.fcs"}
SAMPLE_IDS = {'70397': './data/70397_B_PB_0.fcs',
              '68214': './data/68214_B_PB_0.fcs'}


# Compensation ID to apply (usually extracted from FCS metadata)
# You can override this to use a different compensation matrix
COMP_ID_OVERRIDE = None  # Set to a string to override auto-detected compensation

# ----------------------------------------------------------------------------
# Revision Update Configuration (edit these to control Section 17 behavior)
# ----------------------------------------------------------------------------
# Sample to update in the revision section
SAMPLE_TO_UPDATE = "70397"

# Channel pair to update: specify donor and receiver channel names
# If left as None, the notebook will automatically select the highest off-diagonal
UPDATE_DONOR = "PerCP-Cy5-5-A"
UPDATE_RECEIVER = "PE-A"

# New spillover value to set for the selected donor -> receiver pair
UPDATE_VALUE = 0.5

# ============================================================================
# Verify configuration
# ============================================================================

if not SAMPLE_IDS and not PROJECT_REPO_PATH.exists():
    raise ValueError(
        f"❌ Configuration incomplete:\n"
        f"   - SAMPLE_IDS is empty, but PROJECT_REPO_PATH does not exist\n"
        f"Please either:\n"
        f"   1. Provide SAMPLE_IDS dict with sample_id -> FCS path mapping, OR\n"
        f"   2. Set PROJECT_REPO_PATH to an existing project repository\n"
        f"See 'Setup Instructions' section above for details."
    )

print(f"✓ Project repository: {PROJECT_REPO_PATH.resolve()}")
if SAMPLE_IDS:
    print(f"✓ Sample IDs provided: {list(SAMPLE_IDS.keys())}")
    all_exist = True
    for sid, fcs in SAMPLE_IDS.items():
        fcs_path = Path(fcs)
        all_exists = all_exist and fcs_path.expanduser().exists()
        print(f"   - {sid}: {fcs_path.expanduser().resolve()} Exists: {fcs_path.expanduser().exists()}")

    if not all_exist:
        raise ValueError("❌ One or more FCS file paths in SAMPLE_IDS do not exist. Please check the paths above.")
else:
    print(f"✓ Will use samples from existing project")

✓ Project repository: /home/teo/Repos/cytomind/notebooks/compensation_tutorial
✓ Sample IDs provided: ['70397', '68214']
   - 70397: /home/teo/Projects/flow_pipeline/data/B_PB/70397_B_PB_0.fcs Exists: True
   - 68214: /home/teo/Projects/flow_pipeline/data/B_PB/68214_B_PB_0.fcs Exists: True


## Section 4: Load Project Repository

Initialize the CytoMind pipeline and load the project metadata.

In [4]:
# Initialize the interactive pipeline
rmtree(PROJECT_REPO_PATH, ignore_errors=True)  # Clear existing project for tutorial purposes
pipeline = cytomind.InteractivePipeline(PROJECT_REPO_PATH)

# Load or initialize the project
project = pipeline.repo.load_project()

# If project has no samples but SAMPLE_IDS provided, initialize project with those samples
if len(project.samples) == 0 and SAMPLE_IDS:
    print(f"\n📥 Project not initialized. Running add_samples and load_fcs...")
    print(f"   Samples to add: {list(SAMPLE_IDS.keys())}")

    pipeline.add_samples(SAMPLE_IDS, channel_mapping={})
    print(f"✓ Samples added")

    pipeline.load_fcs()
    print(f"✓ FCS data loaded into AnnData format")

    # Reload project with new data
    project = pipeline.repo.load_project()

print(f"\n✓ Loaded project with {len(project.samples)} samples")
print(f"✓ Compensation matrices: {len(project.compensations)}")
print(f"✓ Channel panel size: {len(project.panel)}")


📥 Project not initialized. Running add_samples and load_fcs...
   Samples to add: ['70397', '68214']
✓ Samples added
✓ FCS data loaded into AnnData format

✓ Loaded project with 2 samples
✓ Compensation matrices: 2
✓ Channel panel size: 13


## Section 5: Inspect Loaded Samples

Display sample references and available compensation matrices.

In [5]:
# Display sample references
print("=" * 80)
print("SAMPLE REFERENCES")
print("=" * 80)
sample_refs = project.samples
for sample_id, ref in sample_refs.items():
    print(f"\n  {sample_id}:")
    print(f"    - FCS file: {ref.fcs}")
    print(f"    - Events: {ref.n_events:,}")
    print(f"    - Default layer: {ref.default_layer}")
    print(f"    - Compensation: {ref.compensation}")

# Use all samples from project for subsequent processing
sample_ids_to_process = list(sample_refs.keys())

print(f"\n✓ Will process {len(sample_ids_to_process)} samples: {sample_ids_to_process}")

# Display available compensation matrices
print("\n" + "=" * 80)
print("AVAILABLE COMPENSATION MATRICES")
print("=" * 80)
for comp_id, comp_ref in project.compensations.items():
    print(f"\n  {comp_id}:")
    print(f"    - Name: {comp_ref.name}")
    print(f"    - Shape: {comp_ref.spill.shape if comp_ref.spill is not None else 'unknown'}")

SAMPLE REFERENCES

  70397:
    - FCS file: /home/teo/Projects/flow_pipeline/data/B_PB/70397_B_PB_0.fcs
    - Events: 1,175,750
    - Default layer: comp
    - Compensation: comp_7518bcad

  68214:
    - FCS file: /home/teo/Projects/flow_pipeline/data/B_PB/68214_B_PB_0.fcs
    - Events: 700,000
    - Default layer: comp
    - Compensation: comp_65261ed4

✓ Will process 2 samples: ['70397', '68214']

AVAILABLE COMPENSATION MATRICES

  comp_7518bcad:
    - Name: 70397_spill
    - Shape: (8, 8)

  comp_65261ed4:
    - Name: 68214_spill
    - Shape: (8, 8)


## Section 6: Apply Compensation

Create a compensation mapping (sample_id → compensation_id) and execute the compensation step.

In [6]:
# Build compensation map: sample_id -> compensation_id
# Use COMP_ID_OVERRIDE if specified, otherwise use sample's default
comp_map = {}
for sample_id in sample_ids_to_process:
    if COMP_ID_OVERRIDE:
        comp_map[sample_id] = COMP_ID_OVERRIDE
    else:
        # Use the compensation ID from sample metadata (if available)
        comp_ref = sample_refs.get(sample_id)
        if comp_ref and comp_ref.compensation:
            comp_map[sample_id] = str(comp_ref.compensation)
        else:
            # Try to use the first available compensation
            if project.compensations:
                comp_map[sample_id] = list(project.compensations.keys())[0]
                print(f"⚠ No compensation found for {sample_id}, using default: {comp_map[sample_id]}")
            else:
                raise ValueError(f"No compensation matrices available for {sample_id}")

print("Compensation mapping:")
for sample_id, comp_id in comp_map.items():
    print(f"  {sample_id} → {comp_id}")


Compensation mapping:
  70397 → comp_7518bcad
  68214 → comp_65261ed4


In [7]:

# Execute the compensation step
print("\nApplying compensation...")
step = pipeline.compensate_samples(comp_map)
print(f"✓ Compensation step completed: {step.id}")
print(f"✓ Step status: {step.status}")


Applying compensation...
✓ Compensation step completed: step_0003
✓ Step status: completed


## Section 7: Review Compensation QC Results

Examine QC metrics from the compensation step to assess spillover correction quality.

In [8]:
# Access QC summary from the completed step
print("=" * 80)
print("COMPENSATION QC SUMMARY")
print("=" * 80)

basic_summary = step.qc_summary["basic_summary"]
print(f"\nOverall QC Status: {basic_summary['overall_flag']}")
print("Per Sample QC Status:")
for sample_id, sample_flag in basic_summary["per_sample_flags"].items():
    print(f"  - {sample_id}: {sample_flag}")

qc_summary = step.qc_summary["detailed_summary"]
qc_summary

COMPENSATION QC SUMMARY

Overall QC Status: PASS
Per Sample QC Status:
  - 70397: PASS
  - 68214: PASS


{'tables': {'channel': 'compensation_tutorial/steps/step_0003/QC/tables/channel_tests.csv',
  'channel_pair': 'compensation_tutorial/steps/step_0003/QC/tables/pairwise_tests.csv'}}

## Section 8: Reload Updated Project

After compensation, reload the project to access the updated dimensions and data layers.

In [9]:
# Reload project to get updated dimensions and layers
project = pipeline.repo.load_project()
dimensions = pipeline.repo.load_dimensions()

print(f"✓ Reloaded project with updated dimensions")
print(f"  Available layers: {list(dimensions.keys())}")

# Check if 'comp' layer was created
if "comp" in dimensions:
    print(f"  ✓ Compensated layer created with {len(dimensions['comp'])} dimensions")
else:
    print(f"  ⚠ No 'comp' layer found (check QC results above)")

✓ Reloaded project with updated dimensions
  Available layers: ['raw', 'comp']
  ✓ Compensated layer created with 13 dimensions


## Section 9: Start Compensation Revision Session

Initiate an interactive revision of the compensation step to adjust spillover coefficients and visualize effects.

In [10]:
# Start a revision session for the compensation step
# This allows interactive adjustment of spillover coefficients
input_spec = {"sample_ids": sample_ids_to_process}

print(f"Starting revision session for step {step.id}...")
handler = pipeline.start_revision(step.id, input_spec=input_spec)

print(f"✓ Revision handler created: {type(handler).__name__}")
print(f"  Workspace: {handler.workspace}")


# Display revision session state
print(f"\nRevision Session:")
print(handler.session)

print("\nSupported Figures:")
print(handler.list_figures(show_args=False))

print("\nSupported Tables:")
print(handler.list_tables(show_args=False))

Starting revision session for step step_0003...
✓ Revision handler created: CompensationRevisionHandler
  Workspace: compensation_tutorial/steps/step_0003/revisions/rev_001

Revision Session:
RevisionSession(id='rev_001', parent_step_id='step_0003', parent_step_type='compensate', state='active', created_at='2026-01-21T14:14:10.667916+00:00', updated_at='2026-01-21T14:14:10.667916+00:00', handler_state={'n_subset': 10000, 'seed': 42, 'fluoro_channels': ['FITC-A', 'PE-A', 'PerCP-Cy5-5-A', 'PE-Cy7-A', 'APC-A', 'APC-H7-A', 'V450-A', 'V500-A'], 'fluoro_markers': ['IgM+IgA', 'IgG+IgA', 'CD45', 'CD19', 'KAPPA', 'LAMBDA', 'CD38', 'CD27'], 'samples': {'70397': {'n_events': 1175750, 'active_compensation': 'comp_7518bcad', 'compensation': 'comp_7518bcad'}, '68214': {'n_events': 700000, 'active_compensation': 'comp_65261ed4', 'compensation': 'comp_65261ed4'}}, 'compensations': {'comp_7518bcad': {'id': 'comp_7518bcad', 'name': '70397_spill', 'source': 'fcs', 'path': 'compensation_tutorial/steps/ste

## Section 10: Inspect Spillover Matrices

Extract and display the current spillover matrices for each sample.

In [11]:
# Select the first sample to inspect
sample_id = SAMPLE_TO_UPDATE

print(f"Spillover matrix for sample '{sample_id}':")
print("=" * 60)

# Get spillover table as a DataFrame
spill_df = handler.get_spillover_table(sample_id)
print(spill_df.round(3))

# Summary statistics
print(f"\nSpillover matrix statistics:")
print(f"  Shape: {spill_df.shape}")
print(f"  Min value: {spill_df.min().min():.4f}")
print(f"  Max value: {spill_df.max().max():.4f}")
print(f"  Mean: {spill_df.values.mean():.4f}")
print(f"  Median: {np.median(spill_df.values):.4f}")

Spillover matrix for sample '70397':
               FITC-A   PE-A  PerCP-Cy5-5-A  PE-Cy7-A  APC-A  APC-H7-A  \
FITC-A          1.000  0.126          0.040     0.004  0.000     0.000   
PE-A            0.018  1.000          0.400     0.028  0.000     0.000   
PerCP-Cy5-5-A   0.000  0.000          1.000     0.151  0.025     0.056   
PE-Cy7-A        0.001  0.009          0.081     1.000  0.000     0.081   
APC-A           0.000  0.000          0.009     0.001  1.000     0.088   
APC-H7-A        0.000  0.000          0.001     0.013  0.023     1.000   
V450-A          0.010  0.006          0.013     0.001  0.000     0.001   
V500-A          0.001  0.000          0.000     0.000  0.000     0.000   

               V450-A  V500-A  
FITC-A          0.001   0.052  
PE-A            0.000   0.000  
PerCP-Cy5-5-A   0.000   0.000  
PE-Cy7-A        0.000   0.000  
APC-A           0.000   0.000  
APC-H7-A        0.000   0.000  
V450-A          1.000   0.256  
V500-A          0.178   1.000  

Spillov

In [12]:
# get position of highest off-diagonal value
spill_df_values = spill_df.values.copy()
np.fill_diagonal(spill_df_values, 0)  # Ignore diagonal
max_pos = np.unravel_index(np.argmax(spill_df_values), spill_df_values.shape)
highest_receiver = spill_df.index[max_pos[0]]
highest_donor = spill_df.columns[max_pos[1]]
donor_channel = UPDATE_DONOR if UPDATE_DONOR is not None else highest_donor
receiver_channel = UPDATE_RECEIVER if UPDATE_RECEIVER is not None else highest_receiver

## Section 11: Visualize Single-Channel Distributions

Plot the distribution of a single fluorescence channel (raw vs. compensated).

In [13]:
# Get available channels (fluorescence channels, not FSC/SSC)
channels = spill_df.columns.tolist()
fluorescence_channels = [ch for ch in channels if not ch.startswith(('FSC', 'SSC'))]

if fluorescence_channels:
    # Select a fluorescence channel to visualize
    channel_name = receiver_channel

    print(f"Visualizing channel: {channel_name}")
    print("=" * 60)

    # Get histogram data from handler
    hist_result = handler.get_figure("channel_histogram", input_params={'sample_id': sample_id, 'channel': channel_name})

    if 'plotly' in hist_result:
        fig = hist_result['plotly']
        fig.show()
    else:
        print("⚠ Histogram visualization not available")
else:
    print("⚠ No fluorescence channels found in the panel")

Visualizing channel: PE-A
Creating viz subset: 70397:raw:10000
Creating raw viz subset from comp layer: 70397:raw:10000


## Section 12: Visualize 2D Spillover Analysis

Create a 2D histogram showing spillover from a donor channel into a receiver channel, and how compensation corrects it.

In [14]:
if len(fluorescence_channels) >= 2:
    # Select two channels to show spillover
    receiver_channel = UPDATE_RECEIVER
    donor_channel = UPDATE_DONOR

    print(f"2D Spillover visualization:")
    print(f"  Receiver: {receiver_channel}")
    print(f"  Donor: {donor_channel}")
    print("=" * 60)

    # Get 2D histogram
    hist2d_result = handler.get_figure("heatmap2d", input_params={'sample_id': sample_id, 'comp_id': 'current', 'receiver': receiver_channel, 'donor': donor_channel})

    if 'plotly' in hist2d_result:
        fig = hist2d_result['plotly']
        fig.show()

        # Print metadata if available
        if 'metadata' in hist2d_result:
            meta = hist2d_result['metadata']
            print(f"\n2D Histogram metadata:")
            for key, value in meta.items():
                print(f"  {key}: {value}")
    else:
        print("⚠ 2D histogram visualization not available")
else:
    print("⚠ Need at least 2 fluorescence channels for 2D spillover analysis")

2D Spillover visualization:
  Receiver: PE-A
  Donor: PerCP-Cy5-5-A



2D Histogram metadata:


## Section 13: Interactive Compensation Tuner

Use the interactive tuner to visualize how adjusting spillover coefficients affects the 2D distribution.

In [15]:
if len(fluorescence_channels) >= 2:
    # Generate an interactive tuner that shows coefficient adjustment effects

    print(f"Generating compensation tuner...")
    print(f"  Sample: {sample_id}")
    print(f"  Receiver: {receiver_channel} (spillover destination)")
    print(f"  Donor: {donor_channel} (spillover source)")
    print("=" * 60)

    # Create a tuner with parameter sweep
    tuner_result = handler.get_figure("heatmap2d_tuner", input_params={'sample_id': sample_id,
                                                                        'comp_id': 'current',
                                                                        'receiver': receiver_channel,
                                                                        'donor': donor_channel})

    if 'plotly' in tuner_result:
        fig = tuner_result['plotly']
        fig.show()

        # Display coefficient range
        if 'metadata' in tuner_result:
            meta = tuner_result['metadata']
            print(f"\nTuner configuration:")
            print(f"  Default coefficient: {meta.get('coef_default', 'unknown'):.4f}")
            print(f"  Min coefficient: {meta.get('coef_min', 'unknown'):.4f}")
            print(f"  Max coefficient: {meta.get('coef_max', 'unknown'):.4f}")
            print(f"  Steps: {meta.get('n_steps', 'unknown')}")
else:
    print("⚠ Need at least 2 channels to create tuner")

Generating compensation tuner...
  Sample: 70397
  Receiver: PE-A (spillover destination)
  Donor: PerCP-Cy5-5-A (spillover source)



Tuner configuration:
  Default coefficient: 0.4000
  Min coefficient: -0.5000
  Max coefficient: 0.5000
  Steps: unknown


## Section 14: Compensation Heatmap Visualization

Display the spillover matrix as an interactive heatmap for visual inspection.

In [16]:
# Create a heatmap of the spillover matrix

fig = handler.get_figure("comp_heatmap", input_params={'sample_id': sample_id})['plotly']
fig.show()

## Section 15: Update Compensation of a Specific Sample

Select the highest off-diagonal spillover coefficient for sample `70397` and set it to `0.5`, then apply the update via the revision handler.

In [ ]:
# Update the compensation for sample 70397 by setting the largest off-diagonal coefficient to 0.5
sample_to_update = SAMPLE_TO_UPDATE

if sample_to_update not in handler.samples:
    print(f"⚠ Sample '{sample_to_update}' is not part of this revision session.")
    print("Available samples:", list(handler.samples.keys()))
else:
    # Get current spillover and ensure numeric
    spill = handler.get_spillover_table(sample_to_update).copy()
    spill = spill.apply(pd.to_numeric, errors="coerce")

    # Determine donor/receiver pair either from config or by auto-selection
    if UPDATE_DONOR is not None and UPDATE_RECEIVER is not None:
        donor = UPDATE_DONOR
        receiver = UPDATE_RECEIVER
        if receiver not in spill.index or donor not in spill.columns:
            raise ValueError(f"Configured donor/receiver not in spillover matrix: donor={donor}, receiver={receiver}")
        old_val = float(spill.at[receiver, donor])
    else:
        M = spill.copy()
        np.fill_diagonal(M.values, np.nan)
        # Guard if all off-diagonal are NaN
        if M.isna().all().all():
            raise ValueError("No off-diagonal coefficients found to update.")
        receiver, donor = M.stack().idxmax()
        old_val = float(spill.at[receiver, donor])

    # Apply update to configured value
    new_val = float(UPDATE_VALUE)
    spill_updated = spill.copy()
    spill_updated.at[receiver, donor] = new_val

    print(f"Updating spillover coefficient {donor} -> {receiver}: {old_val:.4f} -> {new_val:.4f}")

    # Apply revision with updated spillover
    result = handler.apply_revision({
        "sample_id": sample_to_update,
        "spillover": spill_updated,
    })

    print("Revision applied:", {k: result[k] for k in ("status", "mode", "samples_updated", "comp_ids") if k in result})

    # Show updated compensation mapping and a heatmap preview
    current_comp = handler.current_comp(sample_to_update)
    print("Current compensation id:", current_comp)
    fig_info = handler.comp_heatmap(sample_id=sample_to_update, comp_id="current", show_markers=False)
    fig_info["plotly"].show()

Updating spillover coefficient PerCP-Cy5-5-A -> PE-A: 0.4000 -> 0.5000
Revision applied: {'status': 'applied', 'mode': 'new_spillover', 'samples_updated': ['70397'], 'comp_ids': ['comp_f7ee2365']}
Current compensation id: comp_f7ee2365


## Section 16: Commit Revision to Main Repo

Commit the revision changes to the main project repository and re-run downstream compensation for affected samples. Then verify the project reflects the updated compensation.

In [ ]:
# Commit revision changes and reflect them in the main repository

# Commit via handler API (returns metadata updates and optional new step)
step = pipeline.commit_revision(handler)

# Reload project and verify sample 70397 compensation updated
project = pipeline.repo.load_project()
sample_ref = project.samples.get(SAMPLE_TO_UPDATE)
print(f"Sample {SAMPLE_TO_UPDATE} current compensation: {sample_ref.compensation}")
print(f"New step created: {step.id if step else 'no new step'}")

Sample 70397 current compensation: comp_f7ee2365


## Section 17: Troubleshooting & Common Issues

### Issue: "No compensation matrices available"
**Cause:** The FCS files don't contain spillover matrix metadata, or the AddSamplesStep didn't extract them correctly.  
**Solution:** Check that your FCS files have compensation matrices embedded. You can manually create or import a compensation matrix using the project API.

### Issue: "Compensated layer (comp) not created"
**Cause:** The compensation step may have failed QC checks or the spillover matrix is invalid.  
**Solution:** Check the QC summary for specific failures. Verify spillover matrix values are reasonable (typically between 0 and 0.5).

### Issue: Spillover values look too large (e.g., > 1.0)
**Cause:** The compensation matrix may be inverted or scaled incorrectly.  
**Solution:** Review the raw spillover matrix and verify it matches the instrument's calibration.

### Issue: Tuner visualization not loading
**Cause:** Insufficient events or missing data in one of the channels.  
**Solution:** Reduce `n_events` parameter or verify both channels have sufficient signal.

### Modifying Spillover Coefficients

The revision handler allows interactive coefficient adjustment. To programmatically modify a spillover value:

```python
# Example: Adjust a specific spillover coefficient
# (This is a placeholder - implementation depends on handler API)
# handler.set_spillover_coefficient(sample_id, receiver, donor, new_value)
# handler.commit_revision()  # Commit changes back to project
```

For more details, see the CytoMind documentation on revision handlers.

## Section 18: Summary & Next Steps

### What We Accomplished

1. ✓ Loaded an existing CytoMind project with FCS samples
2. ✓ Applied spillover compensation to all samples
3. ✓ Reviewed QC metrics for compensation quality
4. ✓ Inspected spillover matrices across samples
5. ✓ Visualized raw vs compensated data in 1D and 2D
6. ✓ Generated interactive tuning visualizations
7. ✓ Exported results for external review

### Next Steps

- **Refine compensation:** Use the revision handler to adjust spillover coefficients if needed
- **Add transformations:** Create a transformed (log/logicle) layer for downstream analysis
- **QC review:** Examine QC flags and address any warnings
- **Downstream analysis:** Load compensated data in AnnData format for clustering, gating, or statistical analysis

### Output Files

All visualizations have been saved to: `{OUTPUT_DIR}`

### Further Reading

- [CytoMind Documentation](https://github.com/example/cytomind)
- FlowKit FCS parsing: https://github.com/whitman537/flowkit
- Flow Cytometry Compensation Theory: [Flow Cytometry Compensation Tutorial](https://en.wikipedia.org/wiki/Flow_cytometry)

---

**Tutorial Version:** 1.0  
**Last Updated:** 2026-01-20  
**CytoMind Package:** See imports above